<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">GUIDE FOR USING DIFFUSION MODELS FOR LOCAL VIDEO GENERATION.</h2>

<h3 style="color:#44D62C; text-align:left;">Objectives of This Guide</h3>

This guide is designed to walk you through the end-to-end process of generating videos locally using text-to-video diffusion models via the `rzr-aikit` interface.

<h3 style="color:#44D62C; text-align:left;">Project Overview</h3>

AIKit is Razer's AI developer environment built to simplify and accelerate machine learning workflows on high-performance Razer hardware. It offers a plug-and-play experience for running diffusion models and other AI models locally using an optimized stack, enabling fast, private, and fully on-device video generation.

<div style="border-left:4px solid #44D62C; padding:10px 16px; margin:12px 0; background:#111;">
<strong style="color:#44D62C;">Hardware Note:</strong> <span style="color:#FFFFFF;">Video generation is memory-intensive. A GPU with at least <strong>24 GB VRAM</strong> is required. The recommended and validated model for this guide is <code>Wan-AI/Wan2.1-T2V-1.3B-Diffusers</code>.</span>
</div>

<h3 style="color:#44D62C; text-align:left;">🔍 1. Get Model Information</h3>

Use this command to retrieve metadata and configuration details about the Wan2.1 text-to-video model from Hugging Face. This helps you understand its architecture, compatibility, license, and intended use before running it.

In [ ]:
rzr-aikit model info Wan-AI/Wan2.1-T2V-1.3B-Diffusers

<h3 style="color:#44D62C; text-align:left;">📥 2. Download a Model</h3>

Use this command to pull the Wan2.1 text-to-video model from Hugging Face into your local environment.  
This ensures it is available for fast and offline video generation.

In [ ]:
rzr-aikit model download Wan-AI/Wan2.1-T2V-1.3B-Diffusers

<h3 style="color:#44D62C; text-align:left;">🚀 3. Run a Model</h3>

Start a model server to perform local video generation.  
Video diffusion models require the `--omni` flag to use the vLLM-omni engine. Note that the optimization stage is skipped for diffusion models.

In [ ]:
rzr-aikit model run Wan-AI/Wan2.1-T2V-1.3B-Diffusers --omni

<h3 style="color:#44D62C; text-align:left;">🎬 4. Generate a Video</h3>

Once the model is running, you can send a prompt to the `/v1/videos` API endpoint using `curl` and receive a generated video.  
The generated video will be saved as `output.mp4`.

In [ ]:
HOST="http://localhost:8000"
PROMPT="${1:-A dog running on a beach}"
OUTPUT="${2:-output.mp4}"

echo "Submitting job..."
create_response=$(curl -s -X POST "${HOST}/v1/videos" \
  -H "Accept: application/json" \
  -F "prompt=${PROMPT}" \
  -F "width=720" \
  -F "height=480" \
  -F "fps=12" \
  -F "negative_prompt=low quality, blurry, static" \
  -F "num_frames=36")

echo "Response: $create_response"
video_id=$(echo "$create_response" | jq -r '.id')

if [ -z "$video_id" ] || [ "$video_id" = "null" ]; then
  echo "Error: failed to get video ID"
  exit 1
fi

echo "Video ID: $video_id — polling for completion..."

while true; do
  status=$(curl -s "${HOST}/v1/videos/${video_id}" | jq -r '.status')
  echo "Status: $status"
  if [ "$status" = "completed" ]; then
    break
  fi
  if [ "$status" = "failed" ]; then
    echo "Video generation failed"
    exit 1
  fi
  sleep 2
done

echo "Downloading..."
curl -L "${HOST}/v1/videos/${video_id}/content" -o "$OUTPUT"
echo "Saved to $OUTPUT"

<h3 style="color:#44D62C; text-align:left;">▶️ 5. Play the Video</h3>

Play the generated video directly in the notebook using the built-in video player.
<p style="color:red;">Make sure to switch to a Python kernel (Bash[top-right] -> Select Kernel -> Python 3)</p>

In [ ]:
from IPython.display import Video, display

display(Video("output.mp4", embed=True))

<h3 style="color:#44D62C; text-align:left;">🎛️ 6. Advanced: Customize Generation Parameters</h3>

You can fine-tune the video generation by adjusting the request payload. Below are the key settings:

| Parameter | Description | Default / Example |
|---|---|---|
| **prompt** | Describe the video you want to generate | `"A cat walks on the grass, realistic"` |
| **negative_prompt** | Describe what you want to exclude from the output | *(see example above)* |
| **height** | Output frame height in pixels — maximum validated: `480` | `480` |
| **width** | Output frame width in pixels — maximum validated: `720` | `720` |
| **num_frames** | Total number of frames in the output clip | `36` |
| **fps** | Number of frames per second | `12` |
| **guidance_scale** | How closely the output follows the prompt — higher values = more prompt adherence | `5.0` |

<h3 style="color:#44D62C; text-align:left;">🖥️ 7. (Alternative) Use the Web UI for Video Generation</h3>

As an alternative to the command-line interface, you can use the built-in Web UI to generate videos interactively in your browser.  
Once the server is running, open **[http://localhost:7860/](http://localhost:7860/)** to access the interface.

The Web UI exposes all generation parameters for easy control over the output.

In [ ]:
rzr-aikit ui run

<h3 style="color:#44D62C; text-align:left;">🛑 8. Stop the Services</h3>

Stop the Web UI first, then shut down the running model to free GPU and memory resources.

In [ ]:
rzr-aikit ui stop

In [ ]:
rzr-aikit model stop

---

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>

By following this guide, you have successfully learned how to generate videos locally on a Razer-powered system using text-to-video diffusion models via `rzr-aikit`.

##### What You've Achieved

- Retrieved model metadata from Hugging Face
- Downloaded the Wan2.1 text-to-video model into your local environment
- Launched a model server using the vLLM-omni engine
- Generated a video clip by sending a request to the `/v1/videos` API
- Played the generated video directly in the notebook
- Explored the Web UI for interactive, browser-based video generation
- Cleanly stopped the model and released system resources

##### Use Cases This Enables

- Local and offline text-to-video generation for private or secure workloads
- Rapid prototyping and creative experimentation with video diffusion models
- Hardware benchmarking with consistent and repeatable generation configurations
- Interactive exploration via the Web UI without requiring custom tooling

This workflow forms the foundation for AI-powered video generation on Razer hardware. You are now ready to explore more advanced options such as longer clips, higher frame rates, and custom prompting strategies.